# 🎙️ Higgs Audio v3 TTS (4B) — Google Colab

Run [**bosonai/higgs-audio-v3-tts-4b**](https://huggingface.co/bosonai/higgs-audio-v3-tts-4b) end-to-end in Colab with a public **Gradio** interface.

**Features**
- Downloads & installs the model, the [SGLang-Omni](https://github.com/sgl-project/sglang-omni) serving stack, and all dependencies.
- Launches an OpenAI-compatible TTS server (`/v1/audio/speech`) in the background.
- Gradio UI with a **public share link** for:
  - Plain text-to-speech across 100+ languages.
  - **Zero-shot voice cloning** — supply reference audio + its transcript.
  - Inline **control tokens** (emotion / style / prosody / sound effects).
  - **Playback and download** of generated audio.

**Runtime requirements**
- A **GPU runtime** is required: `Runtime → Change runtime type → GPU`.
- The model is ~5B params (BF16, ≈10 GB weights). An **L4 / A100** is recommended; a T4 (16 GB) may work but can be tight.

> ⚠️ **License**: Released under the *Boson Higgs Audio v3 Research and Non-Commercial License*. Voice cloning without consent, impersonation, or any unlawful use is prohibited. Commercial use requires a separate license.

---
**How to use**: run the cells top to bottom. The last cell prints a public `gradio.live` URL.

## 1. Check the GPU

Confirm a GPU is attached. If this errors or shows no GPU, switch the runtime:
`Runtime → Change runtime type → Hardware accelerator: GPU`.

In [ ]:
import subprocess, sys, platform, re

# --- GPU presence ---
try:
    smi = subprocess.check_output(["nvidia-smi"], text=True)
    print(smi)
except Exception as e:
    raise SystemExit(
        "\n❌ No GPU detected. Go to Runtime → Change runtime type → GPU, "
        "then re-run this cell.\n"
    ) from e

# --- Environment report (use this to pick the matching flash-attn wheel) ---
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
plat = "linux_x86_64" if platform.machine() == "x86_64" else platform.machine()

driver_cuda = "?"
m = re.search(r"CUDA Version:\s*([0-9.]+)", smi)
if m:
    driver_cuda = m.group(1)

try:
    nvcc = subprocess.check_output(["nvcc", "--version"], text=True)
    nvcc_cuda = re.search(r"release ([0-9.]+)", nvcc)
    nvcc_cuda = nvcc_cuda.group(1) if nvcc_cuda else "?"
except Exception:
    nvcc_cuda = "not installed"

import torch  # noqa: E402  (Colab default torch until step 2 pins 2.9.1)
abi = getattr(torch._C, "_GLIBCXX_USE_CXX11_ABI", None)

print("=" * 60)
print("ENVIRONMENT (for selecting a flash-attn wheel)")
print("=" * 60)
print(f"Python tag        : {py_tag}   (full: {platform.python_version()})")
print(f"Platform tag      : {plat}")
print(f"GPU               : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a'}")
print(f"Driver CUDA       : {driver_cuda}")
print(f"nvcc CUDA         : {nvcc_cuda}")
print("-" * 60)
print("Current torch (Colab default — will be REPLACED by 2.9.1 in step 2):")
print(f"  torch.__version__   : {torch.__version__}")
print(f"  torch.version.cuda  : {torch.version.cuda}")
print(f"  cxx11 ABI           : {abi}  -> wheel suffix cxx11abi{'TRUE' if abi else 'FALSE'}")
print("=" * 60)
print(
    "\n⚠️  flash-attn must match the torch we install in step 2 (torch==2.9.1, cu12).\n"
    "    Step 2 re-prints this report AFTER pinning torch 2.9.1 — use THAT report's\n"
    "    values (expected: cu12 + torch2.9 + " + py_tag + " + the printed ABI) to pick the wheel from\n"
    "    https://github.com/Dao-AILab/flash-attention/releases"
)

## 2. Install SGLang-Omni and dependencies

Clones [SGLang-Omni](https://github.com/sgl-project/sglang-omni) (provides the `sgl-omni` CLI) and installs its dependencies.

**Why this is the tricky step.** SGLang-Omni's dependencies have a hard conflict under plain pip: `descript-audiotools` pins `protobuf<3.20`, while `sglang`/`onnxruntime`/`s3prl` need `protobuf>=6.31`. The maintainers resolve this with a `[tool.uv].override-dependencies` entry that **only `uv` honours — pip ignores it** (hence `ResolutionImpossible` with pip). So this cell:

1. Pins the **torch stack** (`torch==2.9.1`) first and prints the exact **flash-attn target** (cuda/torch/python/ABI).
2. Installs sglang-omni with **`uv` + a protobuf override**, replicating the maintainers' config.
3. Treats **flash-attn** as optional, and only installs a pasted wheel if it matches the probed torch.

Takes **5–10+ minutes**. On failure, the real error tail is printed at the bottom of the cell (and saved under `/content/log_*.log`).

> To enable flash-attn: copy the printed target line, find the matching wheel at the [flash-attention releases](https://github.com/Dao-AILab/flash-attention/releases), and paste its URL into `FLASH_ATTN_WHEEL` in the code cell.

In [ ]:
import os, re, json, subprocess, sys, shutil

REPO_DIR = "/content/sglang-omni"
PY = sys.executable
LOG_DIR = "/content"


def run_logged(cmd, optional=False, label=""):
    """Run a command, tee output to a log file, surface the error tail on failure."""
    safe = re.sub(r"[^a-zA-Z0-9_.-]+", "_", label or "cmd")
    log_path = os.path.join(LOG_DIR, f"log_{safe}.log")
    print(f"\n$ {' '.join(cmd)}\n  (logging to {log_path})")
    with open(log_path, "w") as lf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            lf.write(line)
        proc.wait()
    if proc.returncode != 0:
        lines = open(log_path).read().splitlines()
        errs = [l for l in lines if re.search(
            r"\b(ERROR|error:|Could not|No matching|conflict|failed building|"
            r"Cannot install|ResolutionImpossible|is incompatible)\b", l, re.I)]
        print("\n" + "=" * 70)
        print(f"❌ FAILED: {label or cmd}  (exit {proc.returncode})")
        print("=" * 70)
        if errs:
            print("\n--- key error lines ---")
            print("\n".join(errs[-40:]))
        print("\n--- last 40 log lines ---")
        print("\n".join(lines[-40:]))
        print(f"\n(Full log: {log_path})")
        if optional:
            print("⚠️  optional — continuing without it.")
            return False
        raise SystemExit(f"Failed: {label}. Copy the lines above.")
    print(f"✅ ok: {label or cmd}")
    return True


def pip(args, **kw):
    return run_logged([PY, "-m", "pip", "install", "-v", *args], **kw)


# 1) Clone the serving stack.
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/sgl-project/sglang-omni.git", REPO_DIR],
                   check=True)
else:
    print("Repo already cloned at", REPO_DIR)

# 2) Pinned torch stack (idempotent).
pip(["torch==2.9.1", "torchvision==0.24.1", "torchaudio==2.9.1"], label="torch")

# 2b) Probe torch + GPU. flash-attn 2.x needs compute capability sm_80+ (Ampere).
import torch  # current kernel torch is fine for capability/ABI reporting
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
sm = cap[0] * 10 + cap[1]
flash_ok = sm >= 80
_probe = (
    "import sys, torch, json;"
    "abi = bool(getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', False));"
    "cu = (torch.version.cuda or '0').split('.')[0];"
    "print(json.dumps({'torch': torch.__version__,"
    "'tmm': '.'.join(torch.__version__.split('.')[:2]),"
    "'cuda_major': cu, 'abi': abi,"
    "'py': 'cp%d%d' % sys.version_info[:2]}))"
)
_info = json.loads(subprocess.check_output([PY, "-c", _probe], text=True).strip())
print("\n--- flash-attn target (torch 2.9.1) ---")
print(f"  torch {_info['torch']} | cuda major cu{_info['cuda_major']} | "
      f"{_info['py']} | cxx11abi{'TRUE' if _info['abi'] else 'FALSE'} | sm_{sm}")
if not flash_ok:
    print(f"  ⚠️  This GPU is sm_{sm} (<sm_80). flash-attn 2.x is NOT supported here —")
    print("      it will be skipped. sglang must use a non-flash attention backend")
    print("      (configure this in the serve cell, step 4).")
else:
    print("  releases: https://github.com/Dao-AILab/flash-attention/releases")
print()

# 3) Install sglang-omni with uv + a protobuf override.
#    pip CANNOT resolve this project (descript-audiotools pins protobuf<3.20 vs
#    sglang/onnxruntime needing >=6.31). The maintainers solve it via uv's
#    [tool.uv].override-dependencies — which pip ignores. So we use uv here.
pip(["uv"], label="uv")
OVERRIDES = "/content/uv_overrides.txt"
with open(OVERRIDES, "w") as f:
    f.write("protobuf>=6.31.1,<7.0.0\n")
run_logged(["uv", "pip", "install", "--system", "--python", PY,
            "--override", OVERRIDES, "-e", REPO_DIR],
           label="sglang-omni")

# 4) flash-attn (OPTIONAL) — only on sm_80+ and only if the wheel matches torch.
FLASH_ATTN_WHEEL = ""  # paste a matching wheel URL (Ampere+ GPUs only)
if not flash_ok:
    print("\nSkipping flash-attn (unsupported on this GPU).")
elif FLASH_ATTN_WHEEL:
    fn = FLASH_ATTN_WHEEL.rsplit("/", 1)[-1]
    want = {
        "cuda": (re.search(r"cu(\d+)", fn) or [None, None])[1],
        "tmm": (re.search(r"torch(\d+\.\d+)", fn) or [None, None])[1],
        "abi": "TRUE" in (re.search(r"cxx11abi(TRUE|FALSE)", fn) or ["", ""])[0],
        "py": (re.search(r"(cp\d+)", fn) or [None, None])[1],
    }
    have = {"cuda": _info["cuda_major"], "tmm": _info["tmm"],
            "abi": _info["abi"], "py": _info["py"]}
    mism = {k: (want[k], have[k]) for k in want if str(want[k]) != str(have[k])}
    if mism:
        print("⚠️  flash-attn wheel does NOT match installed torch — skipping.")
        for k, (w, h) in mism.items():
            print(f"    {k}: wheel={w}  torch={h}")
    else:
        pip([FLASH_ATTN_WHEEL], optional=True, label="flash-attn-wheel")
else:
    print("\nNo FLASH_ATTN_WHEEL set — skipping (sglang will use another backend).")

# 5) UI deps.
pip(["gradio>=4.44", "requests", "soundfile", "huggingface_hub"], label="ui")

# 6) Discover serve options (so we can pick T4-safe flags: dtype, attention backend).
print("\n--- `sgl-omni serve --help` (look for --dtype / --attention-backend) ---")
subprocess.run(["sgl-omni", "serve", "--help"])

print("\nsgl-omni CLI:", shutil.which("sgl-omni") or "NOT FOUND")
print("✅ Install step finished.")

## 3. (Optional) Hugging Face token & download the model

The model weights (~10 GB) are downloaded here. A token is usually **not required** for this public repo, but if the download fails with an auth error, paste a token from <https://huggingface.co/settings/tokens>.

Tip: in Colab you can also store it as a secret named `HF_TOKEN` (🔑 panel on the left) and it will be picked up automatically.

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_ID = "bosonai/higgs-audio-v3-tts-4b"

# Optional token: paste here, or rely on a Colab secret named HF_TOKEN.
HF_TOKEN = ""  # @param {type:"string"}

if not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Using a Hugging Face token.")
else:
    print("No token provided — downloading as a public model.")

print(f"\nDownloading {MODEL_ID} (this can take several minutes)...")
MODEL_PATH = snapshot_download(
    repo_id=MODEL_ID,
    token=HF_TOKEN or None,
    # Skip the large architecture image / assets we don't need at runtime.
    ignore_patterns=["assets/*", "*.png", "*.md"],
)
print("\n✅ Model downloaded to:", MODEL_PATH)

## 4. Launch the TTS server

Starts `sgl-omni serve` in the background on port `8000` and waits until it is ready. Server logs stream to `/content/sgl_omni.log`.

The first launch loads the model into GPU memory and can take a minute or two. If the readiness check times out, inspect the log file (printed at the end) and re-run this cell.

> **T4 note:** the T4 has no native BF16 and no flash-attn. If the server crashes on launch (e.g. a `bfloat16`/attention-backend error in the log), set `EXTRA_SERVE_ARGS` below using the flag names from `sgl-omni serve --help` (printed at the end of step 2) — typically `["--dtype", "float16", "--attention-backend", "triton"]` — and re-run.

In [ ]:
import os, sys, time, shutil, socket, subprocess, importlib.util, requests

HOST = "127.0.0.1"
PORT = 8000
BASE_URL = f"http://{HOST}:{PORT}"
SPEECH_URL = f"{BASE_URL}/v1/audio/speech"
LOG_PATH = "/content/sgl_omni.log"

# Use the locally downloaded snapshot if available, else the repo id.
SERVE_MODEL = globals().get("MODEL_PATH", "bosonai/higgs-audio-v3-tts-4b")

# Extra flags appended to `sgl-omni serve`. On a T4 (sm_75, no BF16, no flash-attn)
# you will likely need to force FP16 and a non-flash attention backend. Fill these
# in using the exact flag names printed by `sgl-omni serve --help` (end of step 2),
# e.g.:  EXTRA_SERVE_ARGS = ["--dtype", "float16", "--attention-backend", "triton"]
EXTRA_SERVE_ARGS = []

# --- Remove TensorFlow before serving ---------------------------------------
# Colab preloads TensorFlow, which bundles its own C++ protobuf runtime. When
# the serve process imports TF (transitively) alongside sglang's grpc/protobuf,
# the C++ descriptor pool aborts with a duplicate-registration crash:
#   "File already exists in database: google/protobuf/duration.proto" -> SIGABRT/-6.
# The TTS serving stack does not need TF, so uninstall it.
if importlib.util.find_spec("tensorflow") is not None:
    print("Removing TensorFlow (avoids the protobuf descriptor crash)...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "tensorflow", "tensorflow-cpu", "tf-nightly"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    print("Done.")
else:
    print("TensorFlow not present — good.")

sgl_omni = shutil.which("sgl-omni")
if sgl_omni is None:
    raise SystemExit("sgl-omni CLI not found — re-run the install cell (step 2).")


def _port_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex((host, port)) == 0


def _server_ready():
    # Try common health/info endpoints; fall back to an open port.
    for path in ("/health", "/v1/models", "/get_model_info"):
        try:
            r = requests.get(BASE_URL + path, timeout=3)
            if r.status_code < 500:
                return True
        except requests.RequestException:
            pass
    return False


# Reuse an already-running server if present.
server_proc = globals().get("server_proc")
if server_proc is not None and server_proc.poll() is None and _port_open(HOST, PORT):
    print("Server already running (pid", server_proc.pid, ").")
else:
    cmd = [sgl_omni, "serve", "--model-path", SERVE_MODEL,
           "--host", HOST, "--port", str(PORT), *EXTRA_SERVE_ARGS]
    print("Launching:", " ".join(cmd))
    # Belt-and-suspenders: also force protobuf's pure-Python implementation.
    server_env = {**os.environ, "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python"}
    log_file = open(LOG_PATH, "w")
    server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT,
                                   env=server_env)
    print(f"Started sgl-omni serve (pid {server_proc.pid}). Waiting for readiness...")

    TIMEOUT_S = 900
    start = time.time()
    while time.time() - start < TIMEOUT_S:
        if server_proc.poll() is not None:
            raise SystemExit(
                f"❌ Server exited early (code {server_proc.returncode}). "
                f"See log: {LOG_PATH}\n\n" + open(LOG_PATH).read()[-3000:]
            )
        if _port_open(HOST, PORT) and _server_ready():
            break
        time.sleep(5)
        print(f"  ...waiting ({int(time.time() - start)}s)")
    else:
        raise SystemExit(
            f"❌ Timed out after {TIMEOUT_S}s. Tail of {LOG_PATH}:\n\n"
            + open(LOG_PATH).read()[-3000:]
        )

print(f"\n✅ Server is up at {BASE_URL}")

## 5. Launch the Gradio interface (public link)

This builds the UI and calls `launch(share=True)`, which prints a public `https://….gradio.live` URL you can open or share.

**Tabs / features**
- **Text-to-Speech** — type text, optionally prepend control tokens, generate, play, and download.
- **Voice cloning** — upload reference audio and paste its transcript; the model clones that voice. Supplying the transcript materially improves fidelity.
- **Control tokens** — pick emotion / style / prosody, or write `<|…|>` tags inline in your text.

In [ ]:
import os, tempfile, requests
import gradio as gr

# --- Control-token vocabularies (from the model card) ---
EMOTIONS = ["(none)", "elation", "amusement", "enthusiasm", "determination",
            "pride", "contentment", "affection", "relief", "contemplation",
            "confusion", "surprise", "awe", "longing", "arousal", "anger",
            "fear", "disgust", "bitterness", "sadness", "shame", "helplessness"]
STYLES = ["(none)", "singing", "shouting", "whispering"]
SPEEDS = ["(none)", "speed_very_slow", "speed_slow", "speed_fast", "speed_very_fast"]
PITCHES = ["(none)", "pitch_low", "pitch_high"]
EXPRESS = ["(none)", "expressive_high", "expressive_low"]


def build_prefix(emotion, style, speed, pitch, express):
    """Delivery tokens shape the whole turn -> placed at the start of `input`."""
    parts = []
    if emotion and emotion != "(none)":
        parts.append(f"<|emotion:{emotion}|>")
    if style and style != "(none)":
        parts.append(f"<|style:{style}|>")
    if speed and speed != "(none)":
        parts.append(f"<|prosody:{speed}|>")
    if pitch and pitch != "(none)":
        parts.append(f"<|prosody:{pitch}|>")
    if express and express != "(none)":
        parts.append(f"<|prosody:{express}|>")
    return "".join(parts)


def synthesize(payload):
    """POST to the local OpenAI-compatible speech endpoint, return a WAV path."""
    resp = requests.post(SPEECH_URL, json=payload, timeout=600)
    ctype = resp.headers.get("content-type", "")
    if resp.status_code != 200 or ctype.startswith("application/json"):
        raise gr.Error(f"Server error ({resp.status_code}): {resp.text[:500]}")
    fd, out_path = tempfile.mkstemp(suffix=".wav", dir="/content")
    with os.fdopen(fd, "wb") as f:
        f.write(resp.content)
    return out_path


def tts_fn(text, emotion, style, speed, pitch, express,
           temperature, top_k, max_new_tokens):
    if not text or not text.strip():
        raise gr.Error("Please enter some text to synthesize.")
    full_input = build_prefix(emotion, style, speed, pitch, express) + text.strip()
    payload = {
        "input": full_input,
        "temperature": float(temperature),
        "top_k": int(top_k),
        "max_new_tokens": int(max_new_tokens),
    }
    return synthesize(payload), full_input


def clone_fn(text, ref_audio, ref_text, emotion, style, speed, pitch, express,
             temperature, top_k, max_new_tokens):
    if not text or not text.strip():
        raise gr.Error("Please enter some text to synthesize.")
    if not ref_audio:
        raise gr.Error("Please upload a reference audio file for voice cloning.")
    full_input = build_prefix(emotion, style, speed, pitch, express) + text.strip()
    reference = {"audio_path": ref_audio}
    if ref_text and ref_text.strip():
        reference["text"] = ref_text.strip()  # improves cloning fidelity
    payload = {
        "input": full_input,
        "references": [reference],
        "temperature": float(temperature),
        "top_k": int(top_k),
        "max_new_tokens": int(max_new_tokens),
    }
    return synthesize(payload), full_input


def control_inputs():
    """Reusable set of control-token + sampling widgets."""
    with gr.Accordion("🎛️ Control tokens (optional)", open=False):
        with gr.Row():
            emotion = gr.Dropdown(EMOTIONS, value="(none)", label="Emotion")
            style = gr.Dropdown(STYLES, value="(none)", label="Style")
        with gr.Row():
            speed = gr.Dropdown(SPEEDS, value="(none)", label="Speed")
            pitch = gr.Dropdown(PITCHES, value="(none)", label="Pitch")
            express = gr.Dropdown(EXPRESS, value="(none)", label="Expressiveness")
        gr.Markdown(
            "You can also type tags inline, e.g. "
            "`<|prosody:pause|>` or `<|sfx:laughter|>Haha`. "
            "Pair every `<|sfx:…|>` with its onomatopoeia."
        )
    with gr.Accordion("⚙️ Sampling parameters", open=False):
        temperature = gr.Slider(0.1, 1.5, value=0.8, step=0.05, label="Temperature")
        top_k = gr.Slider(1, 100, value=50, step=1, label="Top-k")
        max_new_tokens = gr.Slider(128, 4096, value=1024, step=64,
                                   label="Max new tokens")
    return emotion, style, speed, pitch, express, temperature, top_k, max_new_tokens


with gr.Blocks(title="Higgs Audio v3 TTS") as demo:
    gr.Markdown("# 🎙️ Higgs Audio v3 TTS (4B)\nText-to-speech & zero-shot voice cloning.")

    with gr.Tab("Text-to-Speech"):
        tts_text = gr.Textbox(
            label="Text to speak", lines=4,
            placeholder="Type what you want the model to say (100+ languages supported)…",
        )
        ctl = control_inputs()
        tts_btn = gr.Button("🔊 Generate", variant="primary")
        tts_audio = gr.Audio(label="Generated audio", type="filepath",
                             interactive=False)
        tts_sent = gr.Textbox(label="Final input sent to model", interactive=False)
        tts_btn.click(tts_fn, inputs=[tts_text, *ctl],
                      outputs=[tts_audio, tts_sent])

    with gr.Tab("Voice Cloning"):
        gr.Markdown(
            "Upload a short, clean reference clip and paste its **exact transcript** "
            "for best fidelity. Only clone voices you have consent to use."
        )
        clone_ref_audio = gr.Audio(label="Reference audio", type="filepath",
                                   sources=["upload", "microphone"])
        clone_ref_text = gr.Textbox(label="Reference transcript", lines=2,
                                    placeholder="Exact words spoken in the reference clip…")
        clone_text = gr.Textbox(label="Text to speak (in the cloned voice)", lines=4)
        ctl2 = control_inputs()
        clone_btn = gr.Button("🧬 Clone & Generate", variant="primary")
        clone_audio = gr.Audio(label="Generated audio", type="filepath",
                               interactive=False)
        clone_sent = gr.Textbox(label="Final input sent to model", interactive=False)
        clone_btn.click(
            clone_fn,
            inputs=[clone_text, clone_ref_audio, clone_ref_text, *ctl2],
            outputs=[clone_audio, clone_sent],
        )

demo.queue()
demo.launch(share=True)

## 6. Troubleshooting & shutdown

- **No public link?** Gradio prints the `gradio.live` URL in the cell output above — scroll up.
- **Generation errors / 500s** usually come from the server. Inspect the log:
  `print(open("/content/sgl_omni.log").read()[-4000:])`
- **Out of memory** on a T4: switch to an L4/A100 runtime, or lower *Max new tokens*.
- Run the cell below to **stop the server** and free GPU memory when finished.

In [ ]:
# Stop the Gradio app and the TTS server (run when finished).
try:
    demo.close()
    print("Gradio closed.")
except Exception as e:
    print("Gradio not running:", e)

try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print("Server stopped.")
except Exception as e:
    print("Server not running or already stopped:", e)